## Semantic Chunking

In [2]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

In [5]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

model = SentenceTransformer("all-MiniLM-L6-v2")

text = """
Artificial Intelligence is a branch of computer science that focuses on creating systems that can perform tasks that normally require human intelligence. These tasks include reasoning, decision making, problem solving, understanding language, and recognizing images. AI is used in many applications such as virtual assistants, recommendation systems, fraud detection, and autonomous vehicles.

Machine Learning is a subset of Artificial Intelligence that allows computers to learn patterns from data without being explicitly programmed for every task. In supervised learning, models learn from labeled datasets and are commonly used for classification and regression problems. In unsupervised learning, models work with unlabeled data to discover hidden patterns and groups. Reinforcement learning is another approach where an agent learns by interacting with an environment and receiving rewards or penalties.

Deep Learning is a specialized area of Machine Learning that uses artificial neural networks with multiple layers. These networks can learn complex patterns from large amounts of data. Convolutional Neural Networks are commonly used for image-related tasks, while Transformer models have become very important for natural language processing and generative AI applications.

The Eiffel Tower is located in Paris.

France is a popular tourist destination.
"""

sentences = [
    s.strip()
    for s in text.split("\n")
    if s.strip()
]

embeddings = model.encode(sentences)

threshold = 0.7

chunks = []
current_chunk = [sentences[0]]

for i in range(1, len(sentences)):

    sim = cosine_similarity(
        [embeddings[i - 1]],
        [embeddings[i]]
    )[0][0]

    print(
        f"Similarity between sentence {i-1} and {i}: {sim:.2f}"
    )

    if sim > threshold:
        current_chunk.append(sentences[i])

    else:
        chunks.append(" ".join(current_chunk))
        current_chunk = [sentences[i]]

# Last chunk
chunks.append(" ".join(current_chunk))

for i, chunk in enumerate(chunks):
    print(f"\n--- Chunk {i+1} ---")
    print(chunk)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Similarity between sentence 0 and 1: 0.53
Similarity between sentence 1 and 2: 0.47
Similarity between sentence 2 and 3: 0.02
Similarity between sentence 3 and 4: 0.38

--- Chunk 1 ---
Artificial Intelligence is a branch of computer science that focuses on creating systems that can perform tasks that normally require human intelligence. These tasks include reasoning, decision making, problem solving, understanding language, and recognizing images. AI is used in many applications such as virtual assistants, recommendation systems, fraud detection, and autonomous vehicles.

--- Chunk 2 ---
Machine Learning is a subset of Artificial Intelligence that allows computers to learn patterns from data without being explicitly programmed for every task. In supervised learning, models learn from labeled datasets and are commonly used for classification and regression problems. In unsupervised learning, models work with unlabeled data to discover hidden patterns and groups. Reinforcement learning i

### RAG Pipeline Modular Coding

In [6]:
from sentence_transformers import sentence_transformer
from sklearn.metrics.pairwise import cosine_similarity
from langchain_core.documents import Document

import os
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")

In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from langchain_core.documents import Document


class ThresholdSemanticChunker:

    def __init__(
        self,
        model_name="all-MiniLM-L6-v2",
        threshold=0.7
    ):
        self.model = SentenceTransformer(model_name)
        self.threshold = threshold

    def split(self, text: str):

        sentences = [
            s.strip()
            for s in text.split(".")
            if s.strip()
        ]

        if not sentences:
            return []

        embeddings = self.model.encode(sentences)

        chunks = []
        current_chunk = [sentences[0]]

        for i in range(1, len(sentences)):

            sim = cosine_similarity(
                [embeddings[i - 1]],
                [embeddings[i]]
            )[0][0]

            print(
                f"Similarity between sentence {i-1} "
                f"and {i}: {sim:.2f}"
            )

            if sim > self.threshold:
                current_chunk.append(sentences[i])

            else:
                chunks.append(
                    ". ".join(current_chunk) + "."
                )

                current_chunk = [sentences[i]]

        chunks.append(
            ". ".join(current_chunk) + "."
        )

        return chunks

    def split_document(self, docs):

        result = []

        for doc in docs:

            chunks = self.split(doc.page_content)

            for chunk in chunks:

                result.append(
                    Document(
                        page_content=chunk,
                        metadata=doc.metadata
                    )
                )

        return result

In [8]:
sample_text = """"
Artificial Intelligence is a branch of computer science that focuses on creating systems that can perform tasks that normally require human intelligence. These tasks include reasoning, decision making, problem solving, understanding language, and recognizing images. AI is used in many applications such as virtual assistants, recommendation systems, fraud detection, and autonomous vehicles.

Machine Learning is a subset of Artificial Intelligence that allows computers to learn patterns from data without being explicitly programmed for every task. In supervised learning, models learn from labeled datasets and are commonly used for classification and regression problems. In unsupervised learning, models work with unlabeled data to discover hidden patterns and groups. Reinforcement learning is another approach where an agent learns by interacting with an environment and receiving rewards or penalties.

Deep Learning is a specialized area of Machine Learning that uses artificial neural networks with multiple layers. These networks can learn complex patterns from large amounts of data. Convolutional Neural Networks are commonly used for image-related tasks, while Transformer models have become very important for natural language processing and generative AI applications.

The Eiffel Tower is located in Paris.

France is a popular tourist destination.

"""

doc = Document(page_content=sample_text)
doc

Document(metadata={}, page_content='"\nArtificial Intelligence is a branch of computer science that focuses on creating systems that can perform tasks that normally require human intelligence. These tasks include reasoning, decision making, problem solving, understanding language, and recognizing images. AI is used in many applications such as virtual assistants, recommendation systems, fraud detection, and autonomous vehicles.\n\nMachine Learning is a subset of Artificial Intelligence that allows computers to learn patterns from data without being explicitly programmed for every task. In supervised learning, models learn from labeled datasets and are commonly used for classification and regression problems. In unsupervised learning, models work with unlabeled data to discover hidden patterns and groups. Reinforcement learning is another approach where an agent learns by interacting with an environment and receiving rewards or penalties.\n\nDeep Learning is a specialized area of Machin

In [ ]:
chunker = ThresholdSemanticChunker(threhsold=0.7)
chnks=chunker.split_document([doc])
chunks

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Similarity between sentence 0 and 1: 0.47


['Artificial Intelligence is a branch of computer science that focuses on creating systems that can perform tasks that normally require human intelligence. These tasks include reasoning, decision making, problem solving, understanding language, and recognizing images. AI is used in many applications such as virtual assistants, recommendation systems, fraud detection, and autonomous vehicles.',
 'Machine Learning is a subset of Artificial Intelligence that allows computers to learn patterns from data without being explicitly programmed for every task. In supervised learning, models learn from labeled datasets and are commonly used for classification and regression problems. In unsupervised learning, models work with unlabeled data to discover hidden patterns and groups. Reinforcement learning is another approach where an agent learns by interacting with an environment and receiving rewards or penalties.',
 'Deep Learning is a specialized area of Machine Learning that uses artificial neu

# semantic Chunker with Langchain

In [12]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_experimental.text_splitter import SemanticChunker
from langchain_community.document_loaders import TextLoader

In [13]:
loaders = TextLoader("langchain_intro.txt")
docs = loaders.load()

embedding = HuggingFaceEmbeddings()
chunker = SemanticChunker(embedding)

chunks = chunker.split_documents(docs)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

c:\Users\mrraj\Desktop\study\RAG-UDEMY\.venv\Lib\site-packages\huggingface_hub\file_download.py:141: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\mrraj\.cache\huggingface\hub\models--sentence-transformers--all-mpnet-base-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/11.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
print(len(chunks))



2
